# Step 1 — Data load, clean table, and timing audit

This notebook follows the finalized BTP brief exactly for Step 1: load the official NIFTY splits, build one clean chronological table, inspect date gaps, use previous-row market windows only, run the timing assertion, and produce the required EDA outputs.

In [ ]:
!pip -q install datasets pyarrow

import subprocess
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path('/content/btp-regime-aware-multimodal-forecasting')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Hamza-Faarooq/btp-regime-aware-multimodal-forecasting.git', str(REPO_ROOT)], check=True)
sys.path.insert(0, str(REPO_ROOT))

from src.data import load_nifty, build_clean_table, date_gap_report, build_lagged_windows, timing_sanity_check, save_clean_table

## 1. Load the official dataset

The Hugging Face dataset is `raeidsaqur/NIFTY`. Despite the name, its market target is the **SPY index**, not India's NIFTY index. The released dataset has train/valid/test partitions.

In [ ]:
raw = load_nifty()
print('Raw combined shape:', raw.shape)
print('Columns:', raw.columns.tolist())
print('\nSplit sizes:')
print(raw['split'].value_counts().reindex(['train','val','test']))
print('\nDate range:', raw['date'].min().date(), 'to', raw['date'].max().date())
display(raw[['id','date','split','label','pct_change']].head())

# Verify the released split boundaries and the label rule documented by the NIFTY authors.
expected_sizes = {'train': 1477, 'val': 317, 'test': 317}
assert raw['split'].value_counts().to_dict() == expected_sizes
assert raw.loc[(raw['pct_change'] < -0.005), 'label'].eq('Fall').all()
assert raw.loc[(raw['pct_change'] >= -0.005) & (raw['pct_change'] <= 0.005), 'label'].eq('Neutral').all()
assert raw.loc[(raw['pct_change'] > 0.005), 'label'].eq('Rise').all()
print('PASS: official split sizes and ±0.5% label thresholds verified.')

## 2. Build the clean table

The clean table contains date, official split, OHLCV/technical indicators, the day's headline list, the target label, and the supplied daily return.

In [ ]:
clean = build_clean_table(raw)
print('Clean shape:', clean.shape)
print('\nClean columns:')
print(clean.columns.tolist())
print('\nMissing values:')
display(clean.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('\nClass counts by split:')
class_counts = pd.crosstab(clean['split'], clean['label']).reindex(index=['train','val','test'], columns=['Fall','Neutral','Rise'], fill_value=0)
display(class_counts)
print('\nHeadlines per day:')
clean['n_headlines'] = clean['news'].map(len)
display(clean.groupby('split')['n_headlines'].agg(['count','mean','median','min','max']))
save_clean_table(clean)
print('\nSaved data/clean.parquet')

## 3. Date-gap / contiguity check

The brief requires this check before constructing windows. Because the supplied dataset does not contain every SPY trading day in 2010–2020, this implementation uses the brief's permitted **option (b): row-based windows**, and that choice will be stated explicitly in the report.

In [ ]:
gaps = date_gap_report(clean)
gap_counts = gaps['gap_days'].value_counts().sort_index()
print('Unique observed calendar gaps (days):', gap_counts.index.tolist())
print('\nGap frequency:')
display(gap_counts.to_frame('count').head(20))
print('\nLargest gaps:')
display(gaps.nlargest(10, 'gap_days'))
print('\nStep-1 decision: option (b) — define the 10-day market context as the previous 10 dataset rows, not as 10 calendar/trading days.')

## 4. Build the lagged market windows

For target day **t**, the market input is rows **t−10 through t−1**. The target day's own close, high, low, return, and other market values are therefore excluded from its market window.

In [ ]:
X_market, window_meta = build_lagged_windows(clean, lookback=10)
print('Window tensor shape:', X_market.shape)
display(window_meta[['target_date','latest_input_date','split','label']].head())
print('\nRows by target split:')
print(window_meta['split'].value_counts().reindex(['train','val','test']))

## 5. Timing sanity check — mandatory leakage assertion

Every target row must satisfy `latest_input_date < target_date`. If this assertion fails, the pipeline must stop rather than silently continue.

In [ ]:
timing_sanity_check(X_market, window_meta, n_examples=8)

## 6. Required EDA plots

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
class_counts.plot(kind='bar', ax=ax)
ax.set_title('Class counts by official split')
ax.set_xlabel('Split')
ax.set_ylabel('Number of rows')
ax.legend(title='Label')
plt.tight_layout()
Path('figures').mkdir(exist_ok=True)
fig.savefig('figures/step1_class_counts.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plot_df = clean.copy()
plot_df['headline_count'] = plot_df['news'].map(len)
fig, ax = plt.subplots(figsize=(11,5))
ax.plot(plot_df['date'], plot_df['headline_count'], linewidth=0.8)
ax.set_title('Number of headlines per dataset day')
ax.set_xlabel('Date')
ax.set_ylabel('Headline count')
plt.tight_layout()
fig.savefig('figures/step1_headlines_per_day.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 1 checkpoint

Expected outputs:
- `data/clean.parquet`
- `figures/step1_class_counts.png`
- `figures/step1_headlines_per_day.png`
- printed split sizes/date range/columns
- printed date-gap report and the documented option-(b) decision
- printed timing examples and a passing no-lookahead assertion

**Do not move to Step 2 until the timing assertion passes.**